# 07 — OpenClaw Tool Args Categorization

Objectif: analyser les traces OpenClaw et catégoriser les arguments de tool calls en:
- **config_candidate** (plutôt stables)
- **runtime_candidate** (plutôt variables / input utilisateur)
- **mixed**

Hypothèse MVP: **déduplication par nom d'outil uniquement**, puis cette analyse sert à préparer une phase 2 de sous-clustering.


In [ ]:
from __future__ import annotations

import json
import os
import math
from collections import defaultdict, Counter
from pathlib import Path
from statistics import mean

import pandas as pd

try:
    from IPython.display import display as safe_display
except Exception:
    def safe_safe_display(x):
        try:
            print(x.to_string(index=False))
        except Exception:
            print(x)


In [ ]:
# --- Configuration ---
# Ajuste ce chemin si nécessaire
SESSIONS_ROOT = Path.home() / ".openclaw" / "agents"

# Filtre optionnel: mets AGENT_FILTER = None pour tous les agents
AGENT_FILTER = None  # ex: "david"

# Limite optionnelle sur le nombre de fichiers session
MAX_SESSION_FILES = 300


In [ ]:
def list_session_files(root: Path, agent_filter: str | None = None):
    files = []
    if not root.exists():
        return files
    for agent_dir in root.iterdir():
        if not agent_dir.is_dir():
            continue
        agent_id = agent_dir.name
        if agent_filter and agent_id != agent_filter:
            continue
        sessions_dir = agent_dir / "sessions"
        if not sessions_dir.exists():
            continue
        for f in sessions_dir.glob("*.jsonl"):
            if f.name.endswith('.lock') or f.name == 'sessions.json':
                continue
            files.append((agent_id, f))
    files.sort(key=lambda x: x[1].stat().st_mtime, reverse=True)
    return files[:MAX_SESSION_FILES]

session_files = list_session_files(SESSIONS_ROOT, AGENT_FILTER)
len(session_files), session_files[:3]


In [ ]:
def iter_jsonl(path: Path):
    with path.open('r', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue


def normalize_tool_call(message_obj: dict):
    """
    Retourne une liste de (tool_name, args_dict) pour un message assistant
    selon différents formats observés.
    """
    out = []
    msg = message_obj.get("message", {})
    role = msg.get("role")
    if role != "assistant":
        return out

    content = msg.get("content")
    if isinstance(content, list):
        for block in content:
            if not isinstance(block, dict):
                continue
            btype = block.get("type")
            # format observé OpenClaw
            if btype == "toolCall":
                name = block.get("name")
                args = block.get("arguments", {})
                if isinstance(name, str) and isinstance(args, dict):
                    out.append((name, args))
            # autre format possible
            elif btype == "tool_use":
                name = block.get("name")
                args = block.get("input", {})
                if isinstance(name, str) and isinstance(args, dict):
                    out.append((name, args))

    # fallback format OpenAI-compatible (tool_calls)
    tool_calls = msg.get("tool_calls")
    if isinstance(tool_calls, list):
        for tc in tool_calls:
            try:
                name = tc.get("function", {}).get("name")
                raw = tc.get("function", {}).get("arguments", "{}")
                args = json.loads(raw) if isinstance(raw, str) else raw
                if isinstance(name, str) and isinstance(args, dict):
                    out.append((name, args))
            except Exception:
                continue

    return out


def flatten_args(d: dict, parent: str = ""):
    """Flatten dict args en clés dot-notation (a.b.c)."""
    items = {}
    for k, v in d.items():
        key = f"{parent}.{k}" if parent else str(k)
        if isinstance(v, dict):
            items.update(flatten_args(v, key))
        else:
            items[key] = v
    return items


In [ ]:
# Extraction brute des tool calls
rows = []
for agent_id, session_file in session_files:
    session_id = session_file.stem
    for evt in iter_jsonl(session_file):
        if evt.get("type") != "message":
            continue
        for tool_name, args in normalize_tool_call(evt):
            flat = flatten_args(args)
            rows.append({
                "agent_id": agent_id,
                "session_id": session_id,
                "tool": tool_name,
                "args": args,
                "flat_args": flat,
            })

tool_calls_df = pd.DataFrame(rows)
print(f"tool calls extracted: {len(tool_calls_df)}")
tool_calls_df.head(5)


In [ ]:
# Vue globale par outil
by_tool = (
    tool_calls_df.groupby('tool')
    .size()
    .reset_index(name='calls')
    .sort_values('calls', ascending=False)
)
by_tool.head(30)


In [ ]:
def classify_arg(values, n_calls_for_tool):
    """
    Heuristique simple:
    - config_candidate: présence élevée + faible diversité
    - runtime_candidate: diversité élevée (input utilisateur probable)
    - mixed: entre les deux
    """
    non_null = [v for v in values if v is not None]
    present_count = len(non_null)
    if n_calls_for_tool == 0:
        return "mixed", 0.0, 0.0

    presence_rate = present_count / n_calls_for_tool

    norm_values = [json.dumps(v, sort_keys=True, ensure_ascii=False) if isinstance(v, (dict, list)) else str(v) for v in non_null]
    unique_count = len(set(norm_values))
    unique_ratio = unique_count / max(1, present_count)

    # score de variabilité [0,1]
    variability = unique_ratio

    # Heuristiques
    if presence_rate >= 0.9 and unique_ratio <= 0.2:
        cls = "config_candidate"
    elif unique_ratio >= 0.6:
        cls = "runtime_candidate"
    else:
        cls = "mixed"

    return cls, presence_rate, variability


arg_rows = []
for tool, g in tool_calls_df.groupby('tool'):
    n_calls = len(g)
    all_keys = set()
    for fa in g['flat_args']:
        all_keys.update(fa.keys())

    for key in sorted(all_keys):
        vals = [fa.get(key) for fa in g['flat_args']]
        cls, presence_rate, variability = classify_arg(vals, n_calls)

        # exemples valeurs
        shown = []
        for v in vals:
            if v is None:
                continue
            s = json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else str(v)
            if s not in shown:
                shown.append(s)
            if len(shown) >= 3:
                break

        arg_rows.append({
            'tool': tool,
            'arg_key': key,
            'calls_for_tool': n_calls,
            'presence_rate': round(presence_rate, 3),
            'variability': round(variability, 3),
            'classification': cls,
            'sample_values': " | ".join(shown),
        })

arg_df = pd.DataFrame(arg_rows)
arg_df.sort_values(['tool', 'classification', 'arg_key']).head(40)


In [ ]:
# Résumé des classes par outil
summary = (
    arg_df.groupby(['tool', 'classification'])
    .size()
    .reset_index(name='count')
    .pivot(index='tool', columns='classification', values='count')
    .fillna(0)
    .astype(int)
    .reset_index()
)
summary.sort_values('tool')


In [ ]:
# Focus: afficher un outil précis
TOOL_FOCUS = None  # ex: "web_search"

if TOOL_FOCUS:
    safe_display(
        arg_df[arg_df['tool'] == TOOL_FOCUS]
        .sort_values(['classification', 'arg_key'])
        .reset_index(drop=True)
    )
else:
    print("Set TOOL_FOCUS = '<tool_name>' to inspect one tool in detail")


## Notes d'interprétation

- **config_candidate** n'est qu'une suggestion statistique: valider manuellement avant usage en prod.
- En cas de doute, traiter l'arg comme **runtime** (stratégie conservative).
- Cette notebook sert à définir une future allowlist par outil:
  - champs config stables
  - champs runtime utilisateur

Prochaine étape possible: exporter une config YAML/JSON `tool_arg_profiles` depuis ce notebook.


## Per-tool analysis (Top N)

Analyse détaillée sur les `TOP_N_TOOLS` outils les plus appelés.


In [ ]:
# Configuration
TOP_N_TOOLS = 15


def _normalize_sample_value(v):
    if isinstance(v, (dict, list)):
        return json.dumps(v, sort_keys=True, ensure_ascii=False)
    return str(v)


def compute_tool_arg_table(tool_name: str, calls_df: pd.DataFrame) -> pd.DataFrame:
    g = calls_df[calls_df["tool"] == tool_name]
    n_calls = len(g)

    all_keys = sorted({k for fa in g["flat_args"] for k in fa.keys()})
    rows = []

    for key in all_keys:
        vals = [fa.get(key) for fa in g["flat_args"]]
        cls, presence_rate, variability = classify_arg(vals, n_calls)

        uniq = sorted({_normalize_sample_value(v) for v in vals if v is not None})
        sample_values = " | ".join(uniq[:3])

        rows.append({
            "arg_key": key,
            "calls_for_tool": int(n_calls),
            "presence_rate": round(float(presence_rate), 3),
            "variability": round(float(variability), 3),
            "classification": cls,
            "sample_values": sample_values,
        })

    columns = [
        "arg_key",
        "calls_for_tool",
        "presence_rate",
        "variability",
        "classification",
        "sample_values",
    ]
    return pd.DataFrame(rows, columns=columns).sort_values(["classification", "arg_key"]).reset_index(drop=True)


top_tools = (
    by_tool.sort_values(["calls", "tool"], ascending=[False, True])
    .head(TOP_N_TOOLS)["tool"]
    .tolist()
)

tool_arg_tables = {tool_name: compute_tool_arg_table(tool_name, tool_calls_df) for tool_name in top_tools}

print(f"Top tools analysés: {len(top_tools)} / {len(by_tool)}")
top_tools


In [ ]:
# Tableau par outil
for i, tool_name in enumerate(top_tools, start=1):
    calls_for_tool = int(by_tool.loc[by_tool["tool"] == tool_name, "calls"].iloc[0])
    print(f"[{i}/{len(top_tools)}] {tool_name} (calls={calls_for_tool})")
    safe_display(
        tool_arg_tables[tool_name][[
            "arg_key",
            "calls_for_tool",
            "presence_rate",
            "variability",
            "classification",
            "sample_values",
        ]]
    )


In [ ]:
# Résumé + export JSON
EXPORT_PATH = Path("/home/ubuntu/CascadeProjects/AgentCards/lib/vault-exec/notebooks/executed/tool_arg_profiles.json")

tool_arg_profiles = []
for tool_name in top_tools:
    table = tool_arg_tables[tool_name].copy()
    calls_for_tool = int(by_tool.loc[by_tool["tool"] == tool_name, "calls"].iloc[0])

    args_profile = []
    for row in table.to_dict(orient="records"):
        args_profile.append({
            "arg_key": row["arg_key"],
            "calls_for_tool": int(row["calls_for_tool"]),
            "presence_rate": float(row["presence_rate"]),
            "variability": float(row["variability"]),
            "classification": row["classification"],
            "sample_values": row["sample_values"],
        })

    tool_arg_profiles.append({
        "tool": tool_name,
        "calls_for_tool": calls_for_tool,
        "args": args_profile,
    })

payload = {
    "top_n_tools": int(TOP_N_TOOLS),
    "tools_analyzed": int(len(top_tools)),
    "profiles": tool_arg_profiles,
}

EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
EXPORT_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

summary_export = pd.DataFrame([
    {
        "tool": p["tool"],
        "calls_for_tool": p["calls_for_tool"],
        "arg_keys": len(p["args"]),
    }
    for p in tool_arg_profiles
]).sort_values(["calls_for_tool", "tool"], ascending=[False, True]).reset_index(drop=True)

print(f"JSON exporté: {EXPORT_PATH}")
summary_export
